# DiagRAG v9 port arm: ArguAna (argument retrieval), seed 54

Runs the frozen `diagrag_port3.py` (SHA-256 `677a2f38f24a5b0d64849e95930c93ecfd944e28d708753930fde1b2d3a77ada`) on BEIR **arguana** with the settings pre-registered in `docs/evidence/v9_protocol.md`. Kaggle **GPU T4 x1** (or CPU), **Internet On**, Run All.
The gates are port2's, unchanged: a fault passes only if its held-out one-vs-rest AUROC >= 0.70 with >= 20 held-out positives AND its repair has >= 20 hard episodes, a Wilson 95% lower bound >= 0.50 and loses <= 10% of clean successes.
"The suite ports" is claimed only if >= 15 of the 20 ported faults pass both gates. Every fault is reported whichever way it falls. Do not change `SEED` or any setting to shop for a result.
The faults are injected by the project's authors; the code was written by Claude (Anthropic) under the owner's direction. Disclose that in any paper.

In [ ]:
# 1. Pre-registered configuration (docs/evidence/v9_protocol.md). One run per corpus.
import os
CONFIG = dict(DATASET='arguana', SEED='54', N_QUERIES='2500', N_HARD='50', MAX_ATT='2500', N_HARM='20', TIME_BUDGET='36000')
os.environ.update(CONFIG); print(CONFIG)

In [ ]:
# 2. Embedded, hash-pinned code.
EMBED = dict(sha256='677a2f38f24a5b0d64849e95930c93ecfd944e28d708753930fde1b2d3a77ada', size=39539, b64=(
    '...'
))

In [ ]:
# 3. Write the embedded file and verify its SHA-256.
import base64, gzip, hashlib
ON_KAGGLE = os.path.exists('/kaggle/working'); CODE_DIR = '/kaggle/working/code' if ON_KAGGLE else os.path.join(os.getcwd(), 'code'); os.makedirs(CODE_DIR, exist_ok=True)
data = gzip.decompress(base64.b64decode(''.join(EMBED['b64'])))
assert len(data) == EMBED['size'] and hashlib.sha256(data).hexdigest() == EMBED['sha256'] == '677a2f38f24a5b0d64849e95930c93ecfd944e28d708753930fde1b2d3a77ada', 'embedded code is not the frozen port3 file'
open(os.path.join(CODE_DIR, 'diagrag_port3.py'), 'wb').write(data); print('ok diagrag_port3.py', EMBED['sha256'])

In [ ]:
# 4. Run. runpy sets __file__, so PORT_PROTOCOL.json records the hash of the pinned file. If the session stops, run again with this output attached: it resumes from its checkpoint.
import runpy, sys
WORK = '/kaggle/working' if ON_KAGGLE else os.getcwd(); os.environ['DIAGRAG_WORK'] = WORK
os.environ.setdefault('NLTK_DATA', os.path.join(WORK, 'nltk_data')); os.makedirs(os.environ['NLTK_DATA'], exist_ok=True)
try: runpy.run_path(os.path.join(CODE_DIR, 'diagrag_port3.py'), run_name='__main__')
except SystemExit as e: print('stopped with SystemExit', e.code, '- if TIME_BUDGET was reached, run again to resume')

In [ ]:
# 5. Headline. Every fault is in port_solved.csv, pass or fail.
import json, glob
OUT = os.path.join(WORK, 'port3_arguana_out')
vp = os.path.join(OUT, 'PORT_VERDICT.json')
if os.path.exists(vp):
    V = json.load(open(vp)); print({k: V[k] for k in ('domain', 'clean_hit5', 'detectors_pass', 'repairs_pass', 'solved_both', 'of', 'suite_ports', 'not_solved') if k in V})
    P = json.load(open(os.path.join(OUT, 'PORT_PROTOCOL.json'))); print('device', P.get('device'), '| code hash in protocol', P.get('code_sha256'))
else: print('no PORT_VERDICT.json yet: run again to resume')